# KAT-VAD v2.2 — **batch 3 / SC-2: A3-B vs A3-ign** (VideoMAE V2-B in place of V2-S)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§25 (Amendment 14, Y1–Y11)** — it must be in the upload and be
**committed before this notebook trains anything**. Plan: `.project/plans/katvad-v2.2-sc2-a3b.md`.
A3-B = A3-ign (E4-0's recipe) with `v2.motion = V2-B`, `model.motion_dim = 768` — **nothing else changes** (proven in
step 1 before any GPU time). The control is **E4-0's five A3-ign runs**, read from `Thesis-V2/outputs/v2_e4_0/`,
never retrained.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps; refuses an upload without §25 / `sc2_readout` | — |
| 1 | preflight: E4-0 runs + reads, E3 A0 diags, config diff = exactly `{v2.motion, model.motion_dim}`, caches | no |
| 2 | **SC-2a: V2-B on every T2 source** (K's geometry, same cache dir; K's 300 sources are skipped) | **yes** |
| 3 | **bake `A3_R2_B`**: `fit --crn R2 --motion B` on T2, `apply --stride 3` on `dota_cap_dev` (Y1, Y11(c)) | no |
| 4 | **train A3-B × 5** (seeds 2024–2028), chunked + resumable, J10 | **yes** |
| 5 | **diagnostics** (O1′ vs E3's A0 of the same seed) | yes |
| 6 | **protocol B** for A3-B on `dota_cap_dev` | yes |
| 7 | **`sc2_readout`** — the §25 Y8 rule → stop | no |
| 8 | *only if step 7 adopts A3-B (Y10):* DF8 forward — V2-B heads, `score --model A3B`, `same-frame` | yes |

Nothing reads T2-test, DoTA-eval or DoTA-CAP-eval. Every step resumes: re-run the cell after a disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE), v1 outputs
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V22_SC2_ARM'), 'this upload predates SC-2 -- re-upload branch v2'
assert (REPO / 'core/tools/sc2_readout.py').is_file(), 'upload has no core/tools/sc2_readout.py'
ADDENDUM = REPO / 'core/docs/v2/PREREG_ADDENDUM.md'
assert '## 25. Amendment 14' in ADDENDUM.read_text(encoding='utf-8'), 'upload has no addendum §25 (SC-2 rule)'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # 3
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT          # 5
NUM_EPOCHS, CHUNK_EPOCHS = 20, 5                         # E3 / E4-0's recipe
SEEDS      = list(constants.V2_E3_SEEDS)                 # 2024-2028, paired with E4-0's A3-ign
CONTROL    = constants.V22_SC2_CONTROL                   # 'A3ign'
ARM        = constants.V22_SC2_ARM                       # 'A3B'
ANCHOR     = constants.V2_E4_0_ANCHOR_MODE               # 'ignore' (as A3-ign)
CRN        = 'R2'
ENC        = constants.V22_SC2_ENCODER                   # vit_b_k710_dl_from_giant
MOTION_DIM = constants.VIDEOMAE_ARCH[ENC][0]             # 768
STRIDE     = constants.FRAME_STRIDE                      # 8: K's geometry for T2
CAP_SPLIT  = constants.V2_SPLIT_DOTA_CAP_DEV
GROUPS     = ['1-10', '11', '12-42', '43', '44-62']       # CAP-DATA tar groups (step 8 only)
BATCH_GB   = constants.MMAU_STREAM_BATCH_GB

T2        = constants.DATA_ROOT / DATASET
CENSUS    = T2 / 'counts' / 'resolved_census.json'
ORIG      = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
CLIP_T2   = constants.CLIP_CACHE_DIR / DATASET
VIDEO     = constants.VIDEO_CACHE_DIR
DOTA      = constants.DATA_ROOT / 'DoTA'
LABELS    = DOTA / 'labels_s8'                            # defs.json (class names)
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
RAW       = constants.DATA_ROOT / 'MMAU' / 'raw'
ALIGN     = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap' / 'dota_cap_alignment.json'
V2_DATA   = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'
V2_CACHE  = Path(DRIVE_V2) / 'cache'
KNN_V2    = V2_CACHE / 'knn' / f'{DATASET}_v2' / constants.KNN_CACHE_FILENAME
A3_T2     = V2_CACHE / 'v2' / f'A3_{CRN}_S' / DATASET                 # A3 / A3-ign's input (train split ref)
BAKED_T2  = V2_CACHE / 'v2' / f'A3_{CRN}_B' / DATASET                 # A3-B's T2 input (step 3)
CAP_BAKED = V2_CACHE / 'v2' / f'A3_{CRN}_B' / 'DoTA_CAP_s3_from_s1'   # A3-B's DoTA-CAP input (step 3)
E3        = constants.OUTPUT_ROOT / 'v2_e3'               # A0 diags (O1'), p_T2
E40       = constants.OUTPUT_ROOT / 'v2_e4_0'             # control: A3ign/s<seed>, its pb read + diags
SC2       = constants.OUTPUT_ROOT / 'v22_sc2'             # this step: A3B/, REPORTS/
DF8       = constants.OUTPUT_ROOT / 'REPORTS' / 'v22_df8' # step 8 only
LOCAL     = Path('/content/sc2')
for p in (SC2, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('V2_DATA', 'KNN_V2', 'A3_T2', 'BAKED_T2', 'CAP_BAKED', 'E3', 'E40', 'SC2'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 2, 4-6 need one'
df -h /content | tail -1

In [ ]:
import json
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    # Stream a child's output here (and to ``log``); raise with its tail on failure.
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    # Copy a Drive dir (only ``ids`` + its manifests when given) / file to VM disk once, bit-identical (C2).
    t0 = time.time()
    if src.is_dir() and ids is None:
        if not dst.exists():
            shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    elif not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    # Copy a VM-local result to Drive ATOMICALLY (tmp + fsync + rename; lesson (bi)).
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    # The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE checkpoint {path} ({path.stat().st_size / 2**20:.0f} MiB): {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def verified(d):
    # J10: finished, and the checkpoint's own step equals the metrics' last step.
    last = metrics_last(d / 'metrics.jsonl')
    return bool(last and last['epoch'] + 1 == NUM_EPOCHS
                and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'])


def config_diff(a, b):
    out = set()
    for section in set(a) | set(b):
        sa, sb = a.get(section) or {}, b.get(section) or {}
        out |= {f'{section}.{k}' for k in set(sa) | set(sb) if sa.get(k) != sb.get(k)}
    return out

## 1. Preflight — the control, the config diff (Y1, Y11(b)) and the caches, **before** any GPU time

In [ ]:
import tempfile

from core.config import load_config
from core.data.v2_splits import load_split
from core.tools import (build_v2_inputs, dota_cap, e2d_probe, e3_readout, extract_video_features, position_prior,
                        protocol_b_eval, sc2_readout, v2_dataset, v2_diagnostics)

EXPECTED_DIFF = {'v2.motion', 'model.motion_dim'}
cap_dev = load_split(CAP_SPLIT)                                          # sha1-checked
assert len(cap_dev) == 569, len(cap_dev)
for path in (V2_DATA / v2_dataset.MANIFEST_FILENAME, KNN_V2, A3_T2 / constants.V2_INPUT_MANIFEST_FILENAME,
             LABELS / constants.DEFS_FILENAME, ALIGN,
             e3_readout.pb_dir(E40, CONTROL, CAP_SPLIT) / protocol_b_eval.CLIP_SCORES_NPZ,
             E3 / e3_readout.REPORTS / e3_readout.PRIOR_DIR / CAP_SPLIT / position_prior.PRIOR_NPZ):
    assert path.exists(), f'missing {path}'

E40_CFG = {}
for seed in SEEDS:
    d = E40 / CONTROL / f's{seed}'
    assert verified(d), f'E4-0 {CONTROL} s{seed} is not a finished run on Drive (J10)'
    assert (d / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists(), f'E4-0 {CONTROL} s{seed} has no diag'
    a0 = E3 / 'A0' / f's{seed}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    assert a0.exists(), f'missing E3 A0 diag {a0} (O1′ reference)'
    E40_CFG[seed] = yaml.safe_load((d / 'config.yaml').read_text())
    assert E40_CFG[seed]['loss']['dvs_anchor_mode'] == ANCHOR, f'E4-0 {CONTROL} s{seed} is not {ANCHOR}'

SETS = ['train.stage=2', f'train.num_epochs={NUM_EPOCHS}', 'train.amp=true', f'data.dataset={DATASET}',
        f'model.score_head_kernel={KERNEL}', f'loss.mil_topk_pct={TOPK_PCT}', 'kip.enabled=false',
        f'v2.crn={CRN}', f'v2.motion={ENC}', f'model.motion_dim={MOTION_DIM}',
        f'loss.dvs_anchor_mode={ANCHOR}']
for seed in SEEDS:
    with tempfile.TemporaryDirectory() as tmp:     # the config A3-B WILL train with, built now
        cfg = load_config(None, [*SETS, f'train.seed={seed}'])
        cfg.save_yaml(Path(tmp) / 'config.yaml')
        planned = yaml.safe_load((Path(tmp) / 'config.yaml').read_text())
    diff = config_diff(E40_CFG[seed], planned)
    assert diff == EXPECTED_DIFF, f's{seed}: A3-B would differ from E4-0 A3-ign in {diff}'
print(f'every seed: A3-B = E4-0 A3-ign + {sorted(EXPECTED_DIFF)}, nothing else')

sources = build_v2_inputs.t2_all_sources(T2)                             # train + T2-val + test
census = json.loads(CENSUS.read_text())
assert not [v for v in sources if v not in census], 'a T2 source has no census entry'
B_T2 = e2d_probe.t2_video_dir(VIDEO, ENC)                                # K's V2-B cache, extended in step 2
T2_MANIFEST = extract_video_features.build_manifest(ENC, STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC])
e2d_probe.check_manifest(B_T2, T2_MANIFEST)                              # Y11(d): == K's geometry (C2)
alignment_sha = e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)
B_CAP = dota_cap.video_out_dir(ENC)                                      # the DoTA-CAP V2-B cache E2(d) read
e2d_probe.check_manifest(B_CAP, dota_cap.video_manifest(ENC, alignment_sha))
assert all((B_CAP / f'{v}.npy').exists() for v in cap_dev), 'DoTA-CAP V2-B cache incomplete'
done = sum((B_T2 / f'{v}.npy').exists() for v in sources)
print(f'T2 sources {len(sources)} ({done} already V2-B encoded) | dota_cap_dev {len(cap_dev)} | preflight OK')

## 2. SC-2a — V2-B on **every** T2 source (GPU, resumable; Y3)

Same pipeline and geometry as K and as the V2-S extension (`p4_s_full_t2_d6.ipynb` §2): 16 frames every 3rd at the
assumed 30 fps (1.5 s), causal, squash 224², stride 8, into the **same** cache dir as K's 300 V2-B sources (manifest
checked). A shard whose ids are all encoded is skipped. V2-B is ≈ 3–4× V2-S per window. **Re-run this cell after a
disconnect.**

In [ ]:
SHARD, BATCH = 60, 32
root, images = constants.DADA_ORIGIN_ROOT_DIRNAME, constants.DADA_ORIGIN_IMAGES_SUBDIR
assert (ORIG / 'DADA2000.zip').is_file(), f'no DADA2000.zip under {ORIG}'
todo = [v for v in sources if not (B_T2 / f'{v}.npy').exists()]
print(f'{len(todo)} of {len(sources)} T2 sources to encode')


def type_video(vid):
    t, v = vid.split('_')                      # t{type:02d}_v{video:03d}
    return int(t[1:]), int(v[1:])


started = time.monotonic()
for k in range(0, len(todo), SHARD):
    shard, tag = todo[k:k + SHARD], f'{k // SHARD:03d}'
    if all((B_T2 / f'{v}.npy').exists() for v in shard):
        continue
    frames, farm = LOCAL / 'frames', LOCAL / 'flat'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    farm.mkdir(parents=True)
    patterns = [f'{root}/{t}/{v:03d}/{images}/*' for t, v in map(type_video, shard)]
    print(f'--- shard {tag}: {len(shard)} sources, {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')
    run(['7z', 'x', ORIG / 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns])
    for vid in shard:                          # C26: link each source's images dir under its unique id
        t, v = type_video(vid)
        src = frames / root / str(t) / f'{v:03d}' / images
        n = len(list(src.glob('*')))
        if n != census[vid]:                   # C10: a folder is not evidence of data
            raise RuntimeError(f'{vid}: {n} images extracted, census says {census[vid]}')
        (farm / vid).symlink_to(src.resolve(), target_is_directory=True)
    shard_ids = LOCAL / f'ids_{tag}.txt'
    shard_ids.write_text('\n'.join(shard) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_video_features',
         '--frames-dir', farm, '--ids-file', shard_ids, '--dataset', DATASET,
         '--encoder', ENC, '--stride', STRIDE, '--batch-size', BATCH,
         '--device', 'auto', '--output-dir', B_T2], log=LOCAL / 'extract_b.log')
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    encoded {sum((B_T2 / f"{v}.npy").exists() for v in sources)}/{len(sources)} | '
          f'{(time.monotonic() - started) / 60:.1f} min')

assert all((B_T2 / f'{v}.npy').exists() for v in sources), 'full-T2 V2-B cache incomplete -- rerun this cell'
e2d_probe.check_manifest(B_T2, T2_MANIFEST)
bad = [v for v in sources            # one motion row per CLIP step of every source (what `fit` pairs them with)
       if len(np.load(B_T2 / f'{v}.npy', mmap_mode='r')) != len(np.load(CLIP_T2 / f'{v}.npy', mmap_mode='r'))]
assert not bad, f'{len(bad)} sources: V2-B rows != CLIP rows: {bad[:5]}'
EXTRACT_MIN = (time.monotonic() - started) / 60
print(f'V2-B on all {len(sources)} T2 sources: complete, manifest == K geometry, rows == CLIP rows '
      f'({EXTRACT_MIN:.0f} min this session; Y9 prints it)')

## 3. Bake `A3_R2_B` (Y1, Y11(c))

`fit` on the **same** v2 dataset dir as A3 (statistics on T2-train minus T2-val; `train_ids_sha1` must equal A3's);
`apply --stride 3` bakes `dota_cap_dev` from the DoTA-CAP V2-B cache.

In [ ]:
L_V2 = stage(V2_DATA, LOCAL / 'stage' / 'corpus_v2')
L_KNN = stage(KNN_V2, LOCAL / 'stage' / 'knn_v2.npz')
L_T2_OUT, L_CAP_OUT = LOCAL / 'stage' / 'A3B_T2', LOCAL / 'stage' / 'A3B_CAP'
a3 = json.loads((A3_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
if (BAKED_T2 / constants.V2_INPUT_MANIFEST_FILENAME).exists() and \
        (CAP_BAKED / constants.V2_INPUT_MANIFEST_FILENAME).exists():
    L_T2 = stage(BAKED_T2, L_T2_OUT)
    L_CAP = stage(CAP_BAKED, L_CAP_OUT)
else:
    L_CLIP = stage(CLIP_T2, LOCAL / 'stage' / 'clip')
    L_B_T2 = stage(B_T2, LOCAL / 'stage' / 'b_t2', sources)
    L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_cap', cap_dev)
    L_B_CAP = stage(B_CAP, LOCAL / 'stage' / 'b_cap', cap_dev)
    ids_file = LOCAL / 'dota_cap_dev.txt'
    ids_file.write_text(''.join(f'{v}\n' for v in cap_dev))
    if not (L_T2_OUT / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit', '--t2-dir', L_V2, '--clip-dir', L_CLIP,
             '--video-dir', L_B_T2, '--crn', CRN, '--motion', ENC, '--out-dir', L_T2_OUT])
    if not (L_CAP_OUT / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', L_T2_OUT,
             '--clip-dir', L_DOTA_S1, '--video-dir', L_B_CAP, '--stride', constants.V2_E1_STRIDE_BC,
             '--ids-file', ids_file, '--out-dir', L_CAP_OUT])
    L_T2, L_CAP = L_T2_OUT, L_CAP_OUT
m = json.loads((L_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
print(f"A3-B: crn {m['crn']} motion {m['motion']} width {m['width']} s={m['s']:.4f} c={m['c']:.4f} on "
      f"{m['train_sources']} T2-train sources (val {m['t2_val_sources']}); baked {m['baked_ids']}")
assert m['crn'] == CRN and m['motion'] == ENC and m['width'] == 512 + MOTION_DIM, m
assert m['train_ids_sha1'] == a3['train_ids_sha1'], 'A3-B fitted on another train split than A3 -- STOP'
if not (BAKED_T2 / constants.V2_INPUT_MANIFEST_FILENAME).exists():
    publish(L_T2, BAKED_T2)
    publish(L_CAP, CAP_BAKED)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_cap', cap_dev)
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
print('baked + staged')

## 4. Train A3-B × 5 — chunked, resumable, verified (Y1, Y11(a)(b))

E4-0's recipe with V2-B. After every chunk the written `config.yaml` is diffed against E4-0's A3-ign of the same seed
(must be exactly `{v2.motion, model.motion_dim}`). Skip/resume reads the checkpoint's own `global_step` (J10).
**Re-run this cell after a disconnect.**

In [ ]:
def run_dir(seed):
    return SC2 / ARM / f's{seed}'


def train_cmd(seed, out_dir):
    return [sys.executable, '-m', 'core.train', *[x for s in [*SETS, f'train.seed={seed}'] for x in ('--set', s)],
            '--data-dir', L_V2, '--clip-dir', L_T2, '--knn-cache', L_KNN, '--output-dir', out_dir]


def sync(local, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            publish(local / name, drive_dir / name)


def check_diff(seed, cfg_path):
    diff = config_diff(E40_CFG[seed], yaml.safe_load(cfg_path.read_text()))
    assert diff == EXPECTED_DIFF, f'A3-B s{seed} differs from E4-0 A3-ign in {diff} -- STOP'


durations = []
todo = [s for s in SEEDS if not verified(run_dir(s))]
print(f'{len(todo)} runs to train in this session')
for k, seed in enumerate(todo):
    drive_dir, local = run_dir(seed), LOCAL / 'runs' / ARM / f's{seed}'
    local.mkdir(parents=True, exist_ok=True)
    if not (local / 'checkpoint_last.pt').exists() and ckpt_step(drive_dir / 'checkpoint_last.pt') == -1:
        stamp = time.strftime('%Y%m%d-%H%M%S')          # truncated by an interrupted copy: keep, never delete
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            if (drive_dir / name).exists():
                os.replace(drive_dir / name, drive_dir / f'{name}.corrupt-{stamp}')
        print(f'{ARM} s{seed}: Drive checkpoint unreadable -> kept as *.corrupt-{stamp}; retraining (same seed)')
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    t0 = time.time()
    while True:
        done = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done >= NUM_EPOCHS:
            break
        cmd = train_cmd(seed, local)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== {ARM} s{seed}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd, log=LOCAL / f'{ARM}_s{seed}_train.log', quiet_steps=True)
        check_diff(seed, local / 'config.yaml')
        sync(local, drive_dir)
    if not verified(drive_dir):                          # never re-copy a verified checkpoint
        sync(local, drive_dir)
    assert verified(drive_dir), f'{ARM} s{seed}: J10 FAILS on Drive'
    check_diff(seed, drive_dir / 'config.yaml')
    durations.append(time.time() - t0)
    eta = sum(durations) / len(durations) * (len(todo) - k - 1)
    print(f'{ARM} s{seed}: step {metrics_last(drive_dir / "metrics.jsonl")["global_step"]}, '
          f'{durations[-1] / 60:.1f} min; ETA this session {eta / 3600:.1f} h')
missing = [s for s in SEEDS if not verified(run_dir(s))]
print('all 5 A3-B runs verified' if not missing else f'still to train: {missing}')

## 5. Diagnostics — A3-B (O1′ vs E3's A0 of the same seed; Y7)

A3-ign's diags are E4-0's (same code, `t2_val_reads`), never recomputed.

In [ ]:
def diagnose(seed):
    out = SC2 / ARM / f's{seed}' / e3_readout.DIAG_DIR
    if (out / v2_diagnostics.DIAG_JSON).exists():
        return
    a0 = E3 / 'A0' / f's{seed}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    run([sys.executable, '-m', 'core.tools.v2_diagnostics', '--run', f'{ARM}_s{seed}',
         run_dir(seed) / 'checkpoint_last.pt',
         '--data-dir', L_V2, '--t2-input-dir', L_T2, '--dota-input-dir', L_CAP,
         '--dota-s1-dir', L_DOTA_S1, '--dota-data-dir', L_DOTA_LABELS, '--dota-split', CAP_SPLIT,
         '--a0-diag', a0, '--out-dir', out, '--device', 'auto'], log=LOCAL / f'{ARM}_s{seed}_diag.log')


assert not [s for s in SEEDS if not verified(run_dir(s))], 'train every A3-B run first'
for seed in SEEDS:
    diagnose(seed)
print('diagnosed:', sum((SC2 / ARM / f's{s}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists()
                        for s in SEEDS), 'of 5')

## 6. Protocol B for A3-B on `dota_cap_dev` (A3-ign's is E4-0's)

In [ ]:
out = e3_readout.pb_dir(SC2, ARM, CAP_SPLIT)
if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
    print('reusing', out)
else:
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(s) / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs,
         '--input-dir', L_CAP, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', CAP_SPLIT, '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{ARM}_{CAP_SPLIT}_pb.log')
print((out / protocol_b_eval.READOUT_MD).read_text())

## 7. Read-out and the adoption rule (§25 Y8) — no GPU

In [ ]:
run([sys.executable, '-m', 'core.tools.sc2_readout', '--e3-dir', E3, '--e4-dir', E40, '--sc2-dir', SC2,
     '--alignment', ALIGN])
READ = json.loads((SC2 / e3_readout.REPORTS / sc2_readout.READOUT_JSON).read_text())
print((SC2 / e3_readout.REPORTS / sc2_readout.READOUT_MD).read_text())
ADOPTED = READ['decision']['base'] == ARM
print('A3-B ADOPTED -> run step 8' if ADOPTED else 'A3-ign stays -> STOP here (step 8 is not run, Y10)')

## 8. *Only if A3-B was adopted* — DF8 forward (Y10, printed only)

Re-extracts the clamped **V2-B** head windows for the 491 crops (same streaming as `df8_position_shift.ipynb` §3,
gate X10(c)), scores A3-B on every arm (gate X10(b): `full` == step 6's scores), then `same-frame`. Writes
`df8_A3B.*` beside DF8's files; DF8's own read-out is not re-run.

In [ ]:
assert ADOPTED, 'A3-ign stays: step 8 is not run (§25 Y10)'
import pandas as pd

from core.data.v2_inputs import STATS_FILENAME
from core.tools import position_shift

CROPS = DF8 / position_shift.CROPS_JSON
HEADS = DF8 / 'heads_B'
assert CROPS.exists(), 'DF8 crop table missing -- SC-1 must have run'
crops = json.loads(CROPS.read_text())['clips']
alignment = json.loads(ALIGN.read_text())
ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
cap_of = {v: alignment['clips'][v]['cap_id'] for v in crops}
L_B_CAP = stage(B_CAP, LOCAL / 'stage' / 'b_cap', cap_dev)
for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    todo = sorted({cap_of[v] for v in crops if lo <= type_of[cap_of[v]] <= hi
                   and not (HEADS / f'{v}.npz').exists()})
    print(f'group {g}: {len(todo)} CAP videos to stream')
    if not todo:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in todo))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    assert parts, f'no parts for group {g}'
    run([sys.executable, '-m', 'core.tools.position_shift', 'heads', '--crops', CROPS,
         '--alignment', ALIGN, '--parts', *parts, '--cap-ids-file', ids_file, '--video-dir', L_B_CAP,
         '--encoder', ENC, '--work-dir', LOCAL / 'work' / g, '--batch-size', '32',
         '--max-batch-gb', str(BATCH_GB), '--device', 'auto', '--out-dir', HEADS], log=DF8 / 'heads_B_run.log')
report = json.loads((HEADS / position_shift.HEADS_REPORT).read_text())
bad = {v: e for v, e in report.items() if e['reason'] != 'ok'}
assert not bad and all((HEADS / f'{v}.npz').exists() for v in crops), f'X10(c) failures: {list(bad)[:5]}'

L_STATS = LOCAL / 'stage' / 'stats_b'
L_STATS.mkdir(parents=True, exist_ok=True)
for name in (STATS_FILENAME, constants.V2_INPUT_MANIFEST_FILENAME):
    shutil.copy2(BAKED_T2 / name, L_STATS / name)
if not (DF8 / position_shift.scores_json(ARM)).exists():
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(s) / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.position_shift', 'score', '--crops', CROPS, '--model', ARM, *runs,
         '--s1-dir', L_DOTA_S1, '--video-dir', L_B_CAP, '--stats-dir', L_STATS, '--heads-dir', HEADS,
         '--data-dir', L_DOTA_LABELS,
         '--reference', e3_readout.pb_dir(SC2, ARM, CAP_SPLIT) / protocol_b_eval.CLIP_SCORES_NPZ,
         '--device', 'auto', '--out-dir', DF8], log=DF8 / f'score_{ARM}.log')
run([sys.executable, '-m', 'core.tools.position_shift', 'same-frame', '--crops', CROPS, '--scores-dir', DF8,
     '--out-dir', DF8])
print((DF8 / position_shift.SAME_FRAME_MD).read_text())

## 9. Record — then **stop**

Paste `Thesis-V2/outputs/v22_sc2/REPORTS/sc2_readout.md` back to Claude (and, if step 8 ran,
`REPORTS/v22_df8/df8_same_frame.md`). The decision is mechanical (§25 Y8): nothing is re-run, re-seeded or re-tuned
after reading it.